In [ ]:
import os
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GCNConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd
from tqdm import tqdm
import shutil
import concurrent.futures
import threading
from collections import defaultdict
import time

data_lock = threading.Lock()

# Teacher split/initialization seeds are search settings, independent of the
# ten evaluation seeds used by baseline and student training.

def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class GCNTeacher(nn.Module):
    def __init__(self, node_dim: int, global_dim: int, hidden_dims=[128, 128], dropout=0.2):
        super().__init__()
        self.norm = nn.BatchNorm1d(node_dim)
        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h
        self.dropout = nn.Dropout(dropout)
        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, 128), nn.ReLU(), nn.Dropout(dropout)
            )
            self.final_dim = hidden_dims[-1] + 128
        else:
            self.final_dim = hidden_dims[-1]
        self.output = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data: Data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.norm(x)
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = self.dropout(x)
        x = global_mean_pool(x, batch)
        if hasattr(data, 'u') and data.u is not None:
            u = self.global_norm(data.u)
            u = self.global_mlp(u)
            x = torch.cat([x, u], dim=1)
        return self.output(x).squeeze(-1)

def load_graphs(path: str):
    print(f"Loading graph data: {path}")
    graph_path = os.path.join(path, 'graph_data.pt')
    if not os.path.isfile(graph_path):
        raise FileNotFoundError(f"Teacher graph data not found: {graph_path}")
    # Load only trusted files produced by the graph extraction pipeline.
    data = torch.load(graph_path, weights_only=False)
    if not data:
        raise ValueError(f"No graphs found in {graph_path}")
    print(f"Number of samples: {len(data)}  |  Dimensions: Nodes={data[0]['x'].shape[1]}")
    return data

def make_loader(graphs, batch_size, shuffle=True):
    return DataLoader([Data(**g) for g in graphs], batch_size=batch_size, shuffle=shuffle)

def train_and_eval(graphs, split_seed, learn_seed, dropout, batch_size, epochs=200, patience=50):
    print(f"Training parameters => split_seed={split_seed}, learn_seed={learn_seed}, dropout={dropout}, batch_size={batch_size}")
    
    labels = [g['label'].item() for g in graphs]
    
    try:
        idx_train, idx_val = train_test_split(
            np.arange(len(graphs)), 
            test_size=0.1, 
            random_state=split_seed, 
            shuffle=True,
            stratify=labels
        )
        print("✅ Using stratified sampling strategy")
    except ValueError as e:
        print(f"⚠️ Stratified sampling failed: {str(e)}")
        print("⚠️ Switching to random sampling strategy")
        idx_train, idx_val = train_test_split(
            np.arange(len(graphs)), 
            test_size=0.1, 
            random_state=split_seed, 
            shuffle=True
        )
    
    # Each trial gets a new label tensor; never normalize the shared graphs in place.
    train_graphs = [{**graphs[i], 'y': graphs[i]['y'].clone()} for i in idx_train]
    val_graphs = [{**graphs[i], 'y': graphs[i]['y'].clone()} for i in idx_val]
    
    total_samples = len(graphs)
    train_samples = len(train_graphs)
    val_samples = len(val_graphs)
    
    from collections import Counter
    train_label_dist = Counter([g['label'].item() for g in train_graphs])
    val_label_dist = Counter([g['label'].item() for g in val_graphs])
    
    print("\n📊 Dataset Statistics:")
    print(f"Total samples: {total_samples}")
    print(f"Training set samples: {train_samples} ({train_samples/total_samples:.1%})")
    print(f"Validation set samples: {val_samples} ({val_samples/total_samples:.1%})")
    
    print("\n🏷️ Training Set Label Distribution:")
    for label, count in sorted(train_label_dist.items()):
        print(f"  Label {label}: {count} samples ({count/train_samples:.1%})")
    
    print("\n🏷️ Validation Set Label Distribution:")
    for label, count in sorted(val_label_dist.items()):
        print(f"  Label {label}: {count} samples ({count/val_samples:.1%})")
    
    ys = torch.stack([g['y'] for g in train_graphs])
    y_mean, y_std = ys.mean().item(), ys.std().item() + 1e-8
    for g in train_graphs + val_graphs:
        g['y'] = (g['y'] - y_mean) / y_std

    train_loader = make_loader(train_graphs, batch_size)
    val_loader = make_loader(val_graphs, batch_size, shuffle=False)
    
    set_seed(learn_seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    sample = train_graphs[0]
    node_dim = sample['x'].size(1)
    global_dim = sample['u'].size(1) if sample.get('u', None) is not None else 0
    model = GCNTeacher(node_dim, global_dim, dropout=dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()

    best_r2, best_state = -np.inf, None
    no_improve_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses, train_preds, train_trues = [], [], []
        for batch in train_loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = criterion(pred, batch.y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            train_preds.append(pred.detach().cpu().numpy())
            train_trues.append(batch.y.cpu().numpy())
        train_r2 = r2_score(np.concatenate(train_trues), np.concatenate(train_preds))
        train_loss = np.mean(train_losses)

        model.eval()
        val_losses, val_preds, val_trues = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                pred = model(batch)
                val_losses.append(criterion(pred, batch.y).item())
                val_preds.append(pred.cpu().numpy())
                val_trues.append(batch.y.cpu().numpy())
        val_r2 = r2_score(np.concatenate(val_trues), np.concatenate(val_preds))
        val_loss = np.mean(val_losses)

        if epoch % 10 == 0 or epoch == 1 or epoch == epochs:
            print(f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Train R²: {train_r2:.4f} | Val R²: {val_r2:.4f}")

        if val_r2 > best_r2:
            best_r2 = val_r2
            # Capture an independent snapshot; state_dict() alone shares live tensors.
            best_state = copy.deepcopy(model.state_dict())
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"Early stopping triggered: Val R² has no improvement for {patience} consecutive epochs")
                break

    print(f"Best Val R²: {best_r2:.4f}\n")
    return best_r2, best_state, y_mean, y_std, node_dim, global_dim

def generate_predictions(model, graphs, y_mean, y_std, batch_size=64):
    print("Generating predictions...")
    loader = make_loader(graphs, batch_size, shuffle=False)
    device = next(model.parameters()).device
    model.eval()
    normalized_preds = []
    denormalized_preds = []
    
    for batch in tqdm(loader, desc="Prediction Progress"):
        batch = batch.to(device)
        with torch.no_grad():
            p_normalized = model(batch).cpu().numpy()
            p_denormalized = p_normalized * y_std + y_mean
            
            normalized_preds.extend(p_normalized)
            denormalized_preds.extend(p_denormalized)
    
    # Standardized predictions use this particular teacher's y_mean/y_std.
    # Use denormalized predictions before rescaling to a shared student target.
    return np.array(normalized_preds), np.array(denormalized_preds)

def process_feature_directory(feat_dir, predict_dir, output_dir, epochs, split_seeds, learn_seeds, dropouts, batch_sizes, 
                              all_normalized_labels, all_denormalized_labels):
    print(f"\n⏳ Starting to process hierarchical directory: {feat_dir}")
    start_time = time.time()
    
    try:
        graphs = load_graphs(feat_dir)
        feat_name = os.path.basename(feat_dir.rstrip('/\\'))
        feat_output_dir = os.path.join(output_dir, feat_name)
        os.makedirs(feat_output_dir, exist_ok=True)

        best_cfg, best_state, best_mean, best_std = None, None, None, None
        best_r2 = -np.inf
        best_node_dim = None
        best_global_dim = None
        best_hidden_dims = [128, 128]
        best_dropout = None
        
        for ss in split_seeds:
            for ls in learn_seeds:
                for do in dropouts:
                    for bs in batch_sizes:
                        print(f"-- [{feat_name}] Trying configuration: split_seed={ss}, learn_seed={ls}, dropout={do}, batch_size={bs}")
                        r2, state, y_m, y_s, node_dim, global_dim = train_and_eval(
                            graphs, ss, ls, do, bs, epochs
                        )
                        model_name = f"model_ss{ss}_ls{ls}_do{do}_bs{bs}.pt"
                        save_path = os.path.join(feat_output_dir, model_name)
                        
                        torch.save({
                            'model_state_dict': state,
                            'node_dim': node_dim,
                            'global_dim': global_dim,
                            'hidden_dims': [128, 128],
                            'dropout': do,
                            'model_type': 'gcn',
                            'y_mean': y_m,
                            'y_std': y_s
                        }, save_path)
                        
                        if r2 > best_r2:
                            best_r2 = r2
                            best_cfg = (ss, ls, do, bs)
                            best_state = state
                            best_mean = y_m
                            best_std = y_s
                            best_node_dim = node_dim
                            best_global_dim = global_dim
                            best_dropout = do

        ss, ls, do, bs = best_cfg
        best_name = f"{feat_name}_model_ss{ss}_ls{ls}_do{do}_bs{bs}_best.pt"
        best_path = os.path.join(feat_output_dir, best_name)
        
        torch.save({
            'model_state_dict': best_state,
            'node_dim': best_node_dim,
            'global_dim': best_global_dim,
            'hidden_dims': best_hidden_dims,
            'dropout': best_dropout,
            'model_type': 'gcn',
            'y_mean': best_mean,
            'y_std': best_std
        }, best_path)
        
        best_models_dir = os.path.join(output_dir, "best_models")
        os.makedirs(best_models_dir, exist_ok=True)
        best_model_dest = os.path.join(best_models_dir, best_name)
        shutil.copyfile(best_path, best_model_dest)
        print(f"[{feat_name}] Best model saved: {best_path} -> copied to {best_model_dest}")

        graphs_pred = load_graphs(predict_dir)
        
        model = GCNTeacher(
            node_dim=best_node_dim,
            global_dim=best_global_dim,
            hidden_dims=best_hidden_dims,
            dropout=best_dropout
        ).to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
        
        model.load_state_dict(best_state)
        
        normalized_preds, denormalized_preds = generate_predictions(
            model, graphs_pred, best_mean, best_std
        )

        normalized_df = pd.DataFrame({f"{feat_name}_teacher_normalized": normalized_preds})
        normalized_csv = os.path.join(feat_output_dir, 'predict_normalized_labels.csv')
        normalized_df.to_csv(normalized_csv, index=False)
        
        denormalized_df = pd.DataFrame({f"{feat_name}_teacher_denormalized": denormalized_preds})
        denormalized_csv = os.path.join(feat_output_dir, 'predict_denormalized_labels.csv')
        denormalized_df.to_csv(denormalized_csv, index=False)
        
        with data_lock:
            all_normalized_labels[feat_name] = normalized_preds
            all_denormalized_labels[feat_name] = denormalized_preds
            
        print(f"[{feat_name}] ✅ Processing completed! Time elapsed: {time.time()-start_time:.2f} seconds")
        print(f"  Normalized soft labels saved to: {normalized_csv}")
        print(f"  Denormalized soft labels saved to: {denormalized_csv}")
        
        return True, feat_name
    
    except Exception as e:
        print(f"[{feat_dir}] ❌ Processing failed: {str(e)}")
        import traceback
        traceback.print_exc()
        return False, os.path.basename(feat_dir)

def main(base_dir: str, predict_dir: str, output_dir: str, epochs=200):
    os.makedirs(output_dir, exist_ok=True)
    print(f"Root directory of features: {base_dir}")
    print(f"Prediction data directory: {predict_dir}")
    print(f"Output directory: {output_dir}")
    
    if not os.path.isdir(base_dir):
        raise FileNotFoundError(f"Teacher partition directory not found: {base_dir}")
    if not os.path.isdir(predict_dir):
        raise FileNotFoundError(f"Prediction graph directory not found: {predict_dir}")
    feat_dirs = sorted(os.path.join(base_dir, d) for d in os.listdir(base_dir)
                       if os.path.isdir(os.path.join(base_dir, d)))
    if not feat_dirs:
        raise ValueError(f"No teacher partition subdirectories found in {base_dir}")
    num_features = len(feat_dirs)
    # Training trials reseed global PyTorch/NumPy RNGs. Use one worker to
    # prevent concurrent teacher runs from corrupting seed reproducibility.
    print(f"Detected {num_features} teacher partitions; processing sequentially for reproducibility.")
    
    all_normalized_labels = defaultdict(list)
    all_denormalized_labels = defaultdict(list)
    
    # These are internal hyperparameter-search seeds, not evaluation seeds.
    split_seeds = [0, 1, 2]
    learn_seeds = [0, 1, 2]
    dropouts = [0.1, 0.2, 0.3]
    batch_sizes = [32, 64]
    
    successful_features = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future_to_feat = {
            executor.submit(
                process_feature_directory, 
                feat_dir, 
                predict_dir, 
                output_dir, 
                epochs,
                split_seeds,
                learn_seeds,
                dropouts,
                batch_sizes,
                all_normalized_labels,
                all_denormalized_labels
            ): os.path.basename(feat_dir)
            for feat_dir in feat_dirs
        }
        
        with tqdm(total=len(future_to_feat), desc="Parallel Processing Hierarchical Features") as pbar:
            for future in concurrent.futures.as_completed(future_to_feat):
                feat_name = future_to_feat[future]
                try:
                    success, name = future.result()
                    if success:
                        successful_features.append(name)
                    pbar.update(1)
                    pbar.set_postfix_str(f"Completed: {name}")
                except Exception as e:
                    print(f"Error occurred while processing {feat_name}: {str(e)}")
                    pbar.update(1)
    
    # Follow the deterministic partition order rather than future completion order.
    feature_order = [os.path.basename(d) for d in feat_dirs if os.path.basename(d) in all_normalized_labels]
    normalized_df = pd.DataFrame({name: all_normalized_labels[name] for name in feature_order})
    denormalized_df = pd.DataFrame({name: all_denormalized_labels[name] for name in feature_order})
    
    all_normalized_path = os.path.join(output_dir, "all_normalized_labels.csv")
    normalized_df.to_csv(all_normalized_path, index=False)
    
    all_denormalized_path = os.path.join(output_dir, "all_denormalized_labels.csv")
    denormalized_df.to_csv(all_denormalized_path, index=False)
    
    print(f"\n✅ All processing completed! Successfully processed {len(successful_features)}/{num_features} hierarchical features")
    print(f"✅ All normalized soft labels saved to: {all_normalized_path}")
    print(f"✅ All denormalized soft labels saved to: {all_denormalized_path}")
    
    return all_normalized_path, all_denormalized_path

if __name__ == "__main__":
    # Predict on a fixed graph set whose row order is shared across all teachers.
    base_dir = os.path.join("data-set", "teacher_graphs")
    predict_dir = os.path.join("data-set", "train")
    output_dir = os.path.join("checkpoints", "teachers")
    epochs = 300

    normalized_path, denormalized_path = main(base_dir, predict_dir, output_dir, epochs)
    
    print(f"Soft-label generation complete: {normalized_path}, {denormalized_path}")
